# Fire and Smoke Detection ? Roboflow v14

**YOLO Multi-Task Computer Vision Assignment ? Phase 2A**

Prepare and audit the actual fire/smoke detection dataset, then prepare a reproducible nano-model training workflow. **No dataset statistics, predictions, or metrics have been produced in this notebook yet.** All expensive actions are explicitly disabled by default.

Dataset: [gfo-fc1de / fire-and-smoke-detector-yykya / version 14](https://universe.roboflow.com/gfo-fc1de/fire-and-smoke-detector-yykya/dataset/14). Task: bounding-box object detection. Training target: **50 epochs**, **640-pixel image size**, pretrained **YOLO nano**, primarily on **Google Colab GPU**.

The class names are read from the downloaded `data.yaml`; they are not assumed from the project title. Dataset access, license, class list, split counts and quality remain to be verified by the actual export.

## 1. Open and run in Google Colab

1. Download or clone the **complete project**, including `src/` and `requirements.txt`. In Colab choose **File ? Upload notebook** and upload this notebook (or open it from GitHub). Separately upload/unzip the project under `/content/yolo-multi-task-computer-vision`, or clone your own repository there. Opening only the notebook does not upload its supporting files.
2. Select **Runtime ? Change runtime type ? GPU**. Colab controls its Python version; Python 3.11 is the locally validated baseline. Run the next runtime cell and inspect the actual versions.
3. Run the installation cell before the imports cell. Installation is enabled automatically in Colab and is safe to rerun. It preserves the installed Torch/CUDA packages using exact constraints. If a restart is requested, restart and rerun from the top; do not reinstall PyTorch or CUDA. Local Windows already uses this project's `.venv`; do not install into global Python.
4. Run the imports and paths cells. Colab will ask to mount Google Drive; the default output folder is under `MyDrive/YOLO_Multi_Task/fire_smoke_v14`. Local Windows never mounts Drive.
5. Add a Colab secret named `ROBOFLOW_API_KEY` with notebook access, set an environment variable, or enter the key in the hidden prompt. Enable **only** `DOWNLOAD_DATASET` to download version 14. The SDK's output is suppressed because export URLs can contain credentials. Check the dataset license in Roboflow before using it.
6. Inspect the audit table, issue report, actual classes, and annotated samples. Resolve every reported issue before training; do not silently drop invalid samples or move the test set into training.
7. **Stop after dataset preparation for Phase 2A.** The training, validation, and video sections are prepared for later use. Their switches stay False until you explicitly decide to execute them.

Do not publish executed notebooks containing private data, signed URLs, keys, or sensitive paths. Clear outputs before committing. Google Drive checkpoints persist between sessions; `/content` datasets and code do not. After a Colab reset, restore the project, reinstall if needed, download/reconnect the dataset, and rerun the audit. This notebook starts a fresh training run; it does not automatically resume interrupted training.
The first code cell safely clones the complete repository on a fresh Colab runtime, without overwriting an existing project. On local Windows it leaves the current directory unchanged.


In [ ]:
# Colab bootstrap: clone the complete repository before project detection.
from pathlib import Path
import os
import subprocess

if Path("/content").is_dir():
    project_directory = Path("/content/yolo-multi-task-computer-vision")
    required_files = (project_directory / "requirements.txt", project_directory / "src" / "fire_smoke.py")
    if not all(p.is_file() for p in required_files):
        if project_directory.exists():
            raise RuntimeError(
                "Project directory exists but is incomplete. Inspect /content/yolo-multi-task-computer-vision "
                "before retrying; no files were deleted or overwritten."
            )
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Ibraheim11/yolo-multi-task-computer-vision.git",
             str(project_directory)],
            check=True,
        )
    os.chdir(project_directory)
    print("Project ready:", project_directory)
else:
    print("Local runtime: use the existing opened project directory.")


In [ ]:
# Standard-library bootstrap: safe before third-party package installation.
from pathlib import Path
import importlib.util
import os
import platform
import sys

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
IS_WINDOWS = platform.system() == "Windows"
print({"runtime": "Google Colab" if IS_COLAB else platform.system(),
       "python": sys.version.split()[0], "executable": sys.executable})

candidates = [Path.cwd(), Path.cwd().parent]
if IS_COLAB:
    candidates.append(Path("/content/yolo-multi-task-computer-vision"))
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "requirements.txt").is_file()
                     and (p / "src" / "fire_smoke.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Upload/clone the complete project, then change directory to its root and rerun.")
if not IS_COLAB and Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select this project's .venv kernel before running locally.")


## 2. Colab dependency installation ? run before imports

The previous optional install flag defaulted to False, so a fresh Colab runtime could reach `from roboflow import Roboflow` without Roboflow installed. This cell now runs automatically **only in Colab**, using IPython's kernel-aware `%pip` magic through `run_line_magic` (valid Python syntax).

It installs the task's application packages, not the full Windows requirements/lock file. There is no `--upgrade`, `--force-reinstall`, Torch install command, or CUDA install command. Exact constraints protect every installed Torch, torchvision, torchaudio, Triton, and NVIDIA runtime package. If these constraints conflict with an application dependency, resolve that conflict rather than removing the constraints or replacing Colab's working GPU stack. The observed working PyTorch 2.14.1+cu130 / Tesla T4 environment should be retained.

Rerunning uses already satisfied packages. If pip requests a runtime restart (for an already imported library), restart and rerun the setup cells in order. Local Windows skips installation and uses the existing project `.venv`. No datasets, model weights, or training are involved.


In [ ]:
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    if not IS_COLAB:
        raise RuntimeError("Use uv with the existing .venv locally; this installation cell is for Colab.")
    # Standard library only: application imports come after this cell.
    from importlib import metadata
    import shlex
    import tempfile

    def protected_gpu_packages():
        protected = {}
        for distribution in metadata.distributions():
            name = distribution.metadata.get("Name", "").lower().replace("_", "-")
            if name in {"torch", "torchvision", "torchaudio", "triton", "pytorch-triton"} or name.startswith("nvidia-"):
                protected[name] = distribution.version
        return protected

    GPU_PACKAGES_BEFORE = protected_gpu_packages()
    if "torch" not in GPU_PACKAGES_BEFORE:
        raise RuntimeError("Expected Colab's preinstalled PyTorch. Select a standard GPU runtime first.")
    print("Preserving installed GPU packages:", GPU_PACKAGES_BEFORE)
    application_packages = [
        "ultralytics==8.4.208", "roboflow>=1.1,<2",
        "opencv-python>=4.10,<5", "opencv-python-headless>=4.10,<5",
        "PyYAML>=6,<7", "Pillow>=10,<14", "numpy>=1.26,<3",
        "pandas>=2.2,<3", "matplotlib>=3.8,<4", "shapely>=2,<3", "lap>=0.5.12,<0.6",
    ]
    with tempfile.TemporaryDirectory(prefix="yolo-colab-constraints-") as temporary:
        constraints_path = Path(temporary) / "gpu-constraints.txt"
        constraints_path.write_text("".join(f"{name}=={version}\n" for name, version in sorted(GPU_PACKAGES_BEFORE.items())), encoding="utf-8")
        arguments = ["install", "--constraint", str(constraints_path), *application_packages]
        get_ipython().run_line_magic("pip", " ".join(shlex.quote(arg) for arg in arguments))
    if protected_gpu_packages() != GPU_PACKAGES_BEFORE:
        raise RuntimeError("GPU package versions changed unexpectedly. Stop and inspect the installation log.")
    print("Dependency installation cell completed; GPU package versions are unchanged. Verify imports next.")
else:
    print("Local setup: dependency installation skipped; using the existing project .venv.")


## 3. Imports, environment checks, and random seeds

All task imports are grouped here, after installation. Cache/configuration files stay under ignored project outputs. No `YOLO(...)` object is constructed here: constructing a pretrained model can download weights. Fixed seeds improve reproducibility, but GPU kernels, software versions, and hardware can still change results.

In [ ]:
os.environ["YOLO_CONFIG_DIR"] = str(PROJECT_ROOT / "outputs" / ".ultralytics")
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / "outputs" / ".matplotlib")
os.environ["YOLO_AUTOINSTALL"] = "false"
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

import contextlib
from datetime import datetime, timezone
from getpass import getpass
import inspect
import io
import json
import logging
import random

# Explicitly verify every required application import after installation.
from importlib import import_module, metadata
required_imports = {"ultralytics": "ultralytics", "roboflow": "roboflow", "cv2": "opencv-python",
                    "yaml": "PyYAML", "PIL": "Pillow", "numpy": "numpy",
                    "pandas": "pandas", "matplotlib": "matplotlib"}
for module_name, distribution_name in required_imports.items():
    try:
        import_module(module_name)
    except ImportError as exc:
        raise RuntimeError(f"Cannot import {module_name}. Run the dependency installation cell above, inspect pip errors, and restart if requested.") from exc
    print(f"Import OK: {module_name} ({metadata.version(distribution_name)})")

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Video
from roboflow import Roboflow
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ROOT as ULTRALYTICS_ROOT
from ultralytics.utils.downloads import GITHUB_ASSETS_NAMES
import yaml

from src.environment import environment_report, seed_everything
from src.fire_smoke import audit_dataset, locate_yaml, sha256, show_samples, write_training_yaml

SEED = 42
seed_everything(SEED)
VERSIONS = environment_report()
print(json.dumps(VERSIONS, indent=2))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    assert (torch.ones(1, device="cuda") + 1).item() == 2
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
assert torch.equal(torch.tensor([1, 2]) + 1, torch.tensor([2, 3]))


## 4. Colab Drive persistence and local paths

Colab datasets stay on the runtime's local disk for faster reads. Model checkpoints, audit reports, run configuration, validation metrics, and generated videos go to Drive when mounted. Drive mounting is Colab-only and requires authorization in the Colab UI.

`USE_GOOGLE_DRIVE=True` is recommended. If disabled in Colab, output goes to the ephemeral runtime: download it before disconnecting. On Windows, everything remains inside this project. The Roboflow download target is intentionally **not created in advance**: the SDK treats an existing directory as an already-downloaded export.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_FOLDER = "YOLO_Multi_Task/fire_smoke_v14"

if IS_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "fire_smoke"
    if IS_COLAB:
        print("WARNING: outputs are ephemeral without Drive; copy them before runtime shutdown.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_DIR = PROJECT_ROOT / "datasets" / "fire_smoke_v14"
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_ROOT / "training"
AUDIT_DIR = OUTPUT_ROOT / "dataset_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset location:", DATASET_DIR)
print("Persistent output location:" if IS_COLAB and USE_GOOGLE_DRIVE else "Output location:", OUTPUT_ROOT)


## 5. Secure Roboflow version-14 download

Enable the following switch only when ready to download. Prefer Colab Secrets; an environment variable or hidden `getpass` prompt is also supported. No API key is hardcoded, printed, written to configuration, or returned from the helper. The SDK can print signed export URLs, so its output and logging are suppressed, and network errors are replaced with a credential-free message.

The download uses YOLOv8 **detection export format**, which is also compatible with the selected YOLO26 detection model. It does not mean that YOLOv8 weights must be used. The helper does not overwrite existing directories. If a download fails partway, inspect and manually remove or rename the incomplete export before retrying; the notebook never deletes data automatically.

In [ ]:
WORKSPACE = "gfo-fc1de"
PROJECT = "fire-and-smoke-detector-yykya"
DATASET_VERSION = 14
DOWNLOAD_DATASET = False

def download_roboflow_securely(destination):
    if destination.exists():
        raise RuntimeError("Download target already exists. Audit/reuse it, or manually resolve an incomplete export.")
    key = os.environ.get("ROBOFLOW_API_KEY", "").strip()
    if not key and IS_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("ROBOFLOW_API_KEY").strip()
        except Exception:
            key = ""
    if not key:
        key = getpass("Roboflow API key (hidden): ").strip()
    if not key:
        raise RuntimeError("A Roboflow API key is required; no download attempted.")
    previous_logging = logging.root.manager.disable
    try:
        logging.disable(logging.CRITICAL)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            client = Roboflow(api_key=key)
            version = client.workspace(WORKSPACE).project(PROJECT).version(DATASET_VERSION)
            dataset = version.download("yolov8", location=str(destination), overwrite=False)
        return Path(dataset.location).resolve()
    except Exception:
        raise RuntimeError("Roboflow download failed. Check access, identifiers, quota, network, and incomplete files; SDK details are hidden to protect credentials.") from None
    finally:
        key = ""
        logging.disable(previous_logging)

if DOWNLOAD_DATASET:
    DATASET_DIR = download_roboflow_securely(DATASET_DIR)
    print("Version-14 export downloaded. Continue to dataset inspection.")
else:
    print("Download skipped; an existing export can still be inspected.")


## 6. YOLO detection dataset concepts

**Detection and bounding boxes.** Object detection predicts a class and a rectangle for each object instance. Multiple objects can appear in one image. Each YOLO detection label row contains five values: `class_id x_center y_center width height`. Class IDs are zero-based integers; box coordinates are normalized by the image width and height. Width/height must be positive, and the box must fit the image. This is the axis-aligned detection format, not segmentation polygons or classification labels.

**Files and `data.yaml`.** Images normally live in `<split>/images/` and matching annotations in `<split>/labels/`. `example.jpg` maps to `example.txt`. A label file may contain multiple rows. An empty label file can intentionally represent a background image. A missing label is flagged for review here rather than silently assumed to be background. `data.yaml` describes image locations and the ID-to-class-name mapping; it does not contain the image pixels or train the model itself.

**Splits.** Training images update model weights. Validation images guide model selection and hyperparameter choices. Test images provide a final held-out assessment and must not drive tuning. Do not resplit a versioned export without documenting the change. Exact image duplicates across splits are checked by SHA-256; this cannot detect near-duplicates or adjacent video frames, which need manual provenance review.

This inspector handles the directory layout expected from the selected Roboflow detection export. It rejects ambiguous paths, paths outside the export, list/manifest split definitions, missing splits, and malformed names. A missing test split blocks readiness rather than inventing one. Export-relative `../train/images` paths are explicitly resolved only when they identify one existing in-export directory. A clean, absolute-path training YAML is written only after the audit passes; the original export is preserved.

In [ ]:
DATASET_REPORT = None
DATA_YAML = None
if DATASET_DIR.is_dir():
    try:
        DATASET_REPORT = audit_dataset(DATASET_DIR)
    except (ValueError, OSError, yaml.YAMLError) as exc:
        raise RuntimeError("Unable to audit this export. Inspect data.yaml structure and the downloaded directory.") from None
    print("Located YAML:", DATASET_REPORT["source_yaml"])
    print("Actual class names:", DATASET_REPORT["names"])
    summary = pd.DataFrame(DATASET_REPORT["splits"]).T
    display(summary.drop(columns=["class_instances"], errors="ignore"))
    issues = pd.DataFrame(DATASET_REPORT["issues"], columns=["split", "path", "message"])
    print("Audit issues:", len(issues), "| Ready:", DATASET_REPORT["ready"])
    if not issues.empty:
        display(issues.head(50))
        print("First 50 issues shown; the complete issue list is saved in dataset_audit/report.json.")
    (AUDIT_DIR / "report.json").write_text(json.dumps(DATASET_REPORT, indent=2), encoding="utf-8")
    issues.to_csv(AUDIT_DIR / "issues.csv", index=False)
    summary.to_csv(AUDIT_DIR / "split_summary.csv")
    if DATASET_REPORT["ready"]:
        DATA_YAML = write_training_yaml(DATASET_REPORT, AUDIT_DIR / "data.training.yaml")
        print("Audited training YAML:", DATA_YAML)
    else:
        print("Training blocked. Resolve audit issues and rerun; no samples have been removed or modified.")
else:
    print("No dataset is present. Enable download or provide the exact version-14 export before auditing.")


## 7. Representative annotated images

The next cell samples up to two successfully decoded, label-valid images **per split** using seed 42. Red rectangles and text come from the actual annotations and class map. Counts are measured only when a real export is present. The audit checks full image decoding, label readability and format, finite normalized boxes, valid class IDs, image-label pairing, orphan labels, colliding annotation stems, and exact cross-split image duplicates. Review label quality visually even after structural checks pass: a valid rectangle can still be semantically wrong.

Structural errors block training. The report preserves valid rows from partially invalid files for diagnosis, so reported box counts describe parsed rows, not a declaration that the dataset is clean. Sample selection is representative by split, not guaranteed to cover every class.

In [ ]:
if DATASET_REPORT is not None:
    show_samples(DATASET_REPORT, per_split=2, seed=SEED)
else:
    print("Annotated samples unavailable until a real dataset export is audited.")


## 8. Compatible pretrained nano model and training configuration

A nano model offers lower memory use and faster experimentation than larger variants, making it useful for Colab and limited compute. It may miss small, faint, or occluded objects; nano size does not imply accuracy on this dataset. Transfer learning starts from pretrained features, then fine-tunes them for the actual class map instead of learning all features from scratch.

An **epoch** is one pass over the training set; the target here is 50. `imgsz=640` is the training resize target, with aspect ratio handled by Ultralytics; it does not change the meaning of normalized source labels. Higher resolution may help small objects but increases compute and memory. `patience=0` disables early stopping so the requested epoch target is not shortened by patience. Batch size is configurable: lower it if Colab runs out of memory.

The installed package, rather than a latest-version assumption, is checked for the `yolo26n.pt` asset and its matching nano **detection** configuration. This is a compatibility check, not a weight download or a claim that the pretrained checkpoint was run. If the check fails in another runtime, install the validated Ultralytics version and rerun it rather than silently choosing a different task/model.

In [ ]:
MODEL_ID = "yolo26n.pt"
MODEL_CONFIG = ULTRALYTICS_ROOT / "cfg" / "models" / "26" / "yolo26.yaml"
if MODEL_ID not in GITHUB_ASSETS_NAMES or not MODEL_CONFIG.is_file():
    raise RuntimeError("Installed Ultralytics does not list the selected checkpoint/config. Use validated ultralytics==8.4.208.")
model_config = yaml.safe_load(MODEL_CONFIG.read_text(encoding="utf-8"))
assert "n" in model_config["scales"]
assert model_config["head"][-1][2] == "Detect"
assert callable(YOLO.train) and callable(YOLO.val) and callable(YOLO.predict)
print("Verified installed nano detection identifier:", MODEL_ID, "| Ultralytics:", ultralytics.__version__)

EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
ALLOW_LOCAL_TRAINING = False
RUN_NAME = "fire_smoke_v14_seed42_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
TRAINING_CONFIG = dict(epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
                       seed=SEED, deterministic=True, patience=0, workers=0,
                       device=0 if torch.cuda.is_available() else "cpu",
                       project=str(RUNS_DIR), name=RUN_NAME, exist_ok=False,
                       save=True, save_period=10, pretrained=True, plots=True)
print(json.dumps({"model": MODEL_ID, **TRAINING_CONFIG}, indent=2))


## 9. Training ? prepared, not executed in Phase 2A

**Keep `RUN_TRAINING=False` for this phase.** When later authorized, first obtain a clean audit and an active Colab GPU, then enable this flag and execute the cell deliberately. `YOLO(MODEL_ID)` can download pretrained weights only inside this branch. Checkpoints and training artifacts are written directly to the chosen persistent output path. `save_period=10` adds periodic checkpoints; Drive writes may increase runtime. A fresh unique run name prevents overwriting previous experiments.

Training configuration, environment versions, dataset YAML hash, and dataset identity are recorded with the run. A changed Colab GPU/library environment can change results despite seeds. This notebook does not implement automatic resume: after an interruption, inspect the actual saved checkpoints and prepare a separate explicit resume action.

In [ ]:
RUN_TRAINING = False
TRAINING_RUN_DIR = None
if RUN_TRAINING:
    if DATASET_REPORT is None or not DATASET_REPORT["ready"] or DATA_YAML is None:
        raise RuntimeError("A clean dataset audit and generated training YAML are required.")
    if not IS_COLAB and not ALLOW_LOCAL_TRAINING:
        raise RuntimeError("Training defaults to Colab GPU. Local training requires explicit ALLOW_LOCAL_TRAINING=True.")
    if IS_COLAB and not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime before training.")
    if audit_dataset(DATASET_DIR) != DATASET_REPORT:
        raise RuntimeError("Dataset changed after audit. Rerun inspection and review the report before training.")
    planned_run = RUNS_DIR / RUN_NAME
    if planned_run.exists():
        raise RuntimeError("Run directory already exists. Rerun configuration for a fresh run name.")
    provenance = {"dataset": {"workspace": WORKSPACE, "project": PROJECT, "version": DATASET_VERSION},
                  "model": MODEL_ID, "training": TRAINING_CONFIG, "versions": VERSIONS,
                  "source_yaml_sha256": DATASET_REPORT["source_yaml_sha256"],
                  "training_yaml_sha256": sha256(DATA_YAML),
                  "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    (OUTPUT_ROOT / f"{RUN_NAME}.config.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    model = YOLO(MODEL_ID, task="detect")
    train_results = model.train(data=str(DATA_YAML), **TRAINING_CONFIG)
    TRAINING_RUN_DIR = Path(model.trainer.save_dir)
    print("Actual training run:", TRAINING_RUN_DIR)
else:
    print("Training disabled. No pretrained weights loaded; no epochs started.")


## 10. Checkpoints: best.pt and last.pt

`best.pt` contains the checkpoint selected by validation fitness, not by held-out test performance. `last.pt` contains the latest saved training state; interrupted runs may not contain the requested final epoch. They exist only after real training. Set `EXISTING_RUN_DIR` to an actual persistent run directory to inspect an earlier run. File presence alone does not prove successful completion of 50 epochs: inspect training logs and `results.csv`.

In [ ]:
EXISTING_RUN_DIR = None  # Example later: Path("/content/drive/MyDrive/.../training/<actual-run>")
ACTIVE_RUN_DIR = Path(EXISTING_RUN_DIR) if EXISTING_RUN_DIR is not None else TRAINING_RUN_DIR
BEST_PT = ACTIVE_RUN_DIR / "weights" / "best.pt" if ACTIVE_RUN_DIR is not None else None
LAST_PT = ACTIVE_RUN_DIR / "weights" / "last.pt" if ACTIVE_RUN_DIR is not None else None
for label, checkpoint in (("best.pt", BEST_PT), ("last.pt", LAST_PT)):
    print(label, "exists" if checkpoint is not None and checkpoint.is_file() else "not available")


## 11. Validation metrics: Precision, Recall, and mAP

**Precision** measures the fraction of predicted positives that are correct; **recall** measures the fraction of actual objects detected. Both depend on matching and confidence settings. **IoU** measures bounding-box overlap. **AP** summarizes a class precision-recall curve; **mAP50** averages AP across classes at IoU 0.50, while **mAP50-95** averages over IoU thresholds 0.50 through 0.95. Report the split, checkpoint, settings, and actual class names alongside measured values.

The following cell measures validation performance only when enabled and a real `best.pt` exists. It never fills in example metrics. Keep test evaluation separate and run it only after finalizing the model and settings; do not tune from test results. Metrics from a fire/smoke benchmark do not establish safety in real deployment.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    if BEST_PT is None or not BEST_PT.is_file() or DATA_YAML is None:
        raise RuntimeError("A real best.pt and audited training YAML are required for validation.")
    validation_model = YOLO(str(BEST_PT), task="detect")
    validation_results = validation_model.val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE,
                                               device=TRAINING_CONFIG["device"],
                                               project=str(OUTPUT_ROOT / "validation"), name=RUN_NAME)
    measured = {"split": "val", "checkpoint": str(BEST_PT), "precision": float(validation_results.box.mp),
                "recall": float(validation_results.box.mr), "mAP50": float(validation_results.box.map50),
                "mAP50_95": float(validation_results.box.map), "names": validation_results.names}
    print(json.dumps(measured, indent=2))
    (OUTPUT_ROOT / f"{RUN_NAME}.validation.json").write_text(json.dumps(measured, indent=2), encoding="utf-8")
else:
    print("Validation disabled. No measured metrics are available in Phase 2A.")


## 12. Fire/smoke video inference ? prepared for a trained checkpoint

Use a licensed input video and the actual trained `best.pt`; a generic pretrained COCO model is not a substitute for the custom fire/smoke checkpoint. Set the video path below after uploading a video to `assets/videos/` (or a readable Drive path in Colab). The frame loop verifies capture, FPS, dimensions, writer, and every decoded frame; it releases resources on errors. Frames are processed in order without loading the entire video into memory.

The output is an OpenCV MP4 with original FPS and dimensions. OpenCV does not preserve source audio. FPS metadata is required here; variable-frame-rate sources may need a deliberate transcoding policy. A partial video is removed after a processing failure. MP4 playback codec support varies across browsers; download the file if inline playback fails.

In [ ]:
RUN_VIDEO_INFERENCE = False
INPUT_VIDEO = PROJECT_ROOT / "assets" / "videos" / "fire_smoke_input.mp4"
VIDEO_OUTPUT = None
CONFIDENCE = 0.25
if RUN_VIDEO_INFERENCE:
    if BEST_PT is None or not BEST_PT.is_file():
        raise RuntimeError("A real trained best.pt is required for video inference.")
    if not INPUT_VIDEO.is_file():
        raise FileNotFoundError("Upload a video and set INPUT_VIDEO to its actual path.")
    video_model = YOLO(str(BEST_PT), task="detect")
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    writer = None
    candidate_output = OUTPUT_ROOT / "videos" / f"{RUN_NAME}.mp4"
    if candidate_output.exists():
        cap.release()
        raise FileExistsError("Output exists; choose a new run name to preserve it.")
    processed_frames = 0
    try:
        if not cap.isOpened():
            raise RuntimeError("OpenCV cannot open this video.")
        fps = cap.get(cv2.CAP_PROP_FPS)
        expected_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not np.isfinite(fps) or fps <= 0:
            raise RuntimeError("Video FPS is missing/invalid. Inspect or transcode the source first.")
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame is None or frame.size == 0:
                raise RuntimeError("Decoder returned an empty frame.")
            if writer is None:
                height, width = frame.shape[:2]
                candidate_output.parent.mkdir(parents=True, exist_ok=True)
                writer = cv2.VideoWriter(str(candidate_output), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
                if not writer.isOpened():
                    raise RuntimeError("OpenCV cannot create the MP4 output with this codec.")
            if frame.shape[:2] != (height, width):
                raise RuntimeError("Source frame dimensions changed.")
            prediction = video_model.predict(source=frame, imgsz=IMAGE_SIZE, conf=CONFIDENCE,
                                               device=TRAINING_CONFIG["device"], verbose=False)[0]
            writer.write(prediction.plot())
            processed_frames += 1
        if processed_frames == 0:
            raise RuntimeError("No video frames could be decoded.")
        if expected_frames > 0 and processed_frames < expected_frames:
            raise RuntimeError("Decoder stopped before the reported frame count; inspect the source video.")
    except Exception:
        if writer is not None:
            writer.release()
        candidate_output.unlink(missing_ok=True)
        raise
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    if not candidate_output.is_file() or candidate_output.stat().st_size == 0:
        raise RuntimeError("No nonempty video output was produced.")
    VIDEO_OUTPUT = candidate_output
    (OUTPUT_ROOT / f"{RUN_NAME}.video.json").write_text(json.dumps({"source": str(INPUT_VIDEO),
        "source_sha256": sha256(INPUT_VIDEO), "checkpoint": str(BEST_PT), "frames": processed_frames,
        "fps": fps, "confidence": CONFIDENCE, "output": str(VIDEO_OUTPUT)}, indent=2), encoding="utf-8")
    print("Actual output video:", VIDEO_OUTPUT, "| frames:", processed_frames)
else:
    print("Video inference disabled. No output video has been created.")


## 13. Saved output video and playback

This cell displays an output only if the preceding inference cell actually created one. Colab Drive persistence is provided by `OUTPUT_ROOT`. Large videos are not embedded automatically because embedding can make notebooks huge; inspect the path or download/play the file manually. Generated media must remain out of Git.

In [ ]:
if VIDEO_OUTPUT is not None and VIDEO_OUTPUT.is_file():
    print("Saved video:", VIDEO_OUTPUT)
    if VIDEO_OUTPUT.stat().st_size <= 20 * 1024 * 1024:
        display(Video(str(VIDEO_OUTPUT), embed=True))
    else:
        print("Video exceeds the inline preview limit; play/download it from the output folder.")
else:
    print("No saved output video yet.")


## 14. Final observations and limitations ? complete after real execution

No measured result is claimed in this Phase 2A implementation. After execution, document:

- Exact dataset version, classes, measured split counts, label quality, license, and any corrections.
- GPU/runtime versions, checkpoint, training configuration, completed epochs, and actual validation metrics.
- False positives from lights, sunsets, steam, fog or reflections; false negatives for faint smoke, small flames, occlusion, and poor lighting.
- Confidence threshold and representative successes/failures from real video; timing measured on the stated hardware.
- Possible domain shift, class imbalance, label ambiguity, near-duplicate split leakage, and limits of the nano model.

This assignment detector is not a certified fire alarm or a replacement for safety systems. Visual smoke/fire cues can be ambiguous, and benchmark metrics do not guarantee reliable operation in a new location.

## References

- [Required Roboflow dataset version 14](https://universe.roboflow.com/gfo-fc1de/fire-and-smoke-detector-yykya/dataset/14)
- [Ultralytics detection dataset format](https://docs.ultralytics.com/datasets/detect/)
- [Ultralytics training](https://docs.ultralytics.com/modes/train/)
- [Ultralytics validation](https://docs.ultralytics.com/modes/val/)
- [Ultralytics prediction](https://docs.ultralytics.com/modes/predict/)

Installed API and package model assets reviewed for this project on 2026-11-09. Revalidate when changing dependencies. **Phase 2A stops before the 50-epoch training run.**